In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def V(x):
    return (-1 -np.exp (-((x+15) /10) **2) \
    -2*np.exp (-((x -10) /10) **2) + \
    -np.exp (-((x -35) /10) **2) \
    -3*np.exp (-((x -60) /10) **2) \
    -np.exp (-((x -85) /10) **2)\
    -2*np.exp (-((x -110) /10) **2) \
    ) * \
    (1+1/5* np.cos (2* np.pi /2.5*x))

In [ ]:
xs = [np.linspace(-1,11, 1000), np.linspace(-5,105,1000)]
fig, axes = plt.subplots(2,1, figsize=(16,6))

for x, ax in zip(xs, axes):
    ax.plot(x,V(x))
    ax.grid()

In [ ]:
class Walker():
    def __init__(self, x=None, kT=0.05, Nsteps=1000):
        if x is None:
            x = 100 * np.random.rand()
        self.x = x
        self.kT = kT
        self.balls = {}
        self.Nsteps = Nsteps
        self.steps = 0
        self.xs = None
        self.steps_until_solution = None

    def step(self):
        new_x = self.x + np.random.randn()
        dE = V(new_x) - V(self.x)
        if dE < 0 or np.random.rand() < np.exp(-dE/self.kT):
            self.x = new_x%100
        self.steps += 1

    def optim(self, Nsim=10000):
        self.xs = []
        for i in range(Nsim):
            self.step()
            self.xs.append(self.x)

    def draw(self,ax,i=None,force_create_drawing_object=False):
        if self.balls.get(ax,None) is None or force_create_drawing_object:
            self.balls[ax] = ax.plot(self.x,V(self.x),'o',ms=10,alpha=0.5)[0]
        ball = self.balls[ax]
        if i is None:
            x = self.x
        else:
            if self.xs is None:
                self.optim()
            i = min(max(0,i),len(self.xs)-1)
            x = self.xs[i]
        ball.set_data([x],[V(x)])
        return ball



In [ ]:
fig, axes = plt.subplots(2,2,figsize=(12,5))
for column,kT in enumerate([0.1, 3]):
    xs = np.linspace(-1, 11, 1000)
    axes[1,column].plot(xs,V(xs))
    walker = Walker(3,kT=kT)
    walker.draw(axes[1,column])
    xs = [walker.x]
    for _ in range(8):
        walker.step()
        xs.append(walker.x)
        walker.draw(axes[1,column],force_create_drawing_object=True)
    axes[0,column].plot(xs,range(len(xs)),color='k')
    axes[0,column].scatter(xs,range(len(xs)),c=[f'C{i+1}' for i in range(len(xs))])
    for ax in axes.ravel():
        ax.set_xlim([-1,11])
    axes[0,column].set_ylabel('Monte Carlo time')
    axes[1,column].set_ylabel('Potential Energy')
    axes[1,column].set_xlabel('x')

In [ ]:
# configs
temperatures = [0.1, 3.0]
n_steps = 8
x_bounds = (-1, 11)
xs_plot = np.linspace(*x_bounds, 1000)

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 5), sharex=True)

for col, kT in enumerate(temperatures):
    ax_time = axes[0, col]
    ax_potential = axes[1, col]
    
    walker = Walker(3, kT=kT) # init walker and tracking
    trajectory = [walker.x]

    walker.draw(ax_potential) # init state
    
    for _ in range(n_steps): # do MC
        walker.step()
        trajectory.append(walker.x)
        walker.draw(ax_potential, force_create_drawing_object=True)

    mc_times = range(len(trajectory))
    colors = [f"C{i+1}" for i in mc_times]

    ax_time.plot(trajectory, mc_times, color="k", zorder=1) # upper panel, time trajec
    ax_time.scatter(trajectory, mc_times, c=colors, zorder=2)
    ax_time.set_ylabel("Monte Carlo time")
    ax_time.set_title(f"kT = {kT}")    
    ax_potential.plot(xs_plot, V(xs_plot), color="tab:blue") # lower panel, energy landscape
    ax_potential.set_ylabel("Potential Energy")
    ax_potential.set_xlabel("x")

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
xs = np.linspace(-5, 105, 1000)
ax.plot(xs,V(xs))

from matplotlib import animation
plt.rc('animation', html='jshtml')

walkers = [Walker(kT=.5) for _ in range(5)]

N = 500
Nbefore = 10
Nafter = 10
def update(i):

    if i < Nbefore:
        i -= Nbefore
    else:
        i *= 10
    all_balls = []
    for walker in walkers:
        ball = walker.draw(ax,i-Nbefore)
        all_balls.append(ball)
    return all_balls

anim = animation.FuncAnimation(fig,
                               update,
                               frames=Nbefore + N + Nafter,
                               interval=50,
                               blit=True)
anim

